<!-- source: new + slide 56–57 -->
# M6 · MCP, bezpieczeństwo i dalszy rozwój

**Przebieg:** prezentacja, demo, lab

> *„Agent odpowiada w notebooku. Ale nasze narzędzia żyją w wielu miejscach: w Unity Catalog, w Genie, w wyszukiwarce, a jutro w systemach spoza Databricks. Czy każdy agent musi mieć do nich własne integracje?”* — CTO, TechRetail Corp

**Model Context Protocol (MCP)** to jeden standard zamiast N × M integracji: każdy zgodny klient rozmawia z każdym zgodnym serwerem. Dla agenta narzędzie MCP wygląda identycznie jak funkcja Unity Catalog z M2. Zmienia się tylko **źródło** narzędzi.

| Część | Co robisz | Lab |
|---|---|---|
| 1 | lista narzędzi z zarządzanego serwera MCP funkcji UC | wywołanie narzędzia |
| 2 | agent LangGraph z narzędziami z trzech serwerów MCP | demo |
| 3 | ryzyka agentów na żywo: „zbuduj i złam” (Mariusz buduje, Krzysztof atakuje), sześć warstw obrony, least privilege | — |
| 4 | ograniczenia Free Edition, co dodać przed PoC i produkcją, zamknięcie dnia | — |

**Free Edition:** zarządzane serwery MCP są w Public Preview. Jeśli w Twoim workspace nie odpowiadają, komórki wypiszą powód i obejrzysz demo prowadzącego. Nic dalej od nich nie zależy.

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
# source: WS3[2]
dbutils.library.restartPython()

In [ ]:
# source: new + WS4[3] + WS2[6]
# Wspólna konfiguracja warsztatu — ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

In [ ]:
# source: new + WS4[18]
import asyncio

from databricks.ai_search.client import AISearchClient
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
HOST = w.config.host.rstrip("/")
USERNAME = spark.sql("SELECT current_user()").first()[0]
VIP_CUSTOMER_ID = int(spark.table(GOLD_TABLE).where("loyalty_segment = 3 AND num_orders > 0 AND city IS NOT NULL AND tax_id IS NOT NULL").orderBy("customer_id").first()["customer_id"])
TRY_MCP = True  # ustaw False, żeby pominąć wywołania MCP (np. gdy Preview jest wyłączony)

try:
    SEARCH_READY = bool(AISearchClient(disable_notice=True).get_index(SEARCH_ENDPOINT, SEARCH_INDEX).describe().get("status", {}).get("ready"))
except Exception:
    SEARCH_READY = False
GENIE_SPACE_ID = next((s.space_id for s in (w.genie.list_spaces().spaces or []) if s.title == GENIE_TITLE), None)

MCP_SERVERS = {"funkcje UC": f"{HOST}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}"}
if SEARCH_READY:
    MCP_SERVERS["AI Search"] = f"{HOST}/api/2.0/mcp/vector-search/{CATALOG}/{SCHEMA}"
if GENIE_SPACE_ID:
    MCP_SERVERS["Genie Agent"] = f"{HOST}/api/2.0/mcp/genie/{GENIE_SPACE_ID}"


def run_async(coroutine):
    # asyncio.run nie działa, gdy pętla zdarzeń już biegnie (część środowisk notebookowych)
    try:
        return asyncio.run(coroutine)
    except RuntimeError:
        import nest_asyncio

        nest_asyncio.apply()
        return asyncio.get_event_loop().run_until_complete(coroutine)


for label, url in MCP_SERVERS.items():
    print(f"{label:<12} {url.replace(HOST, '<workspace>')}")

<!-- source: slide 58 + slide 59 + WS4[17] -->
## 1. MCP na Databricks: agent jako klient, narzędzia jako serwery

| Rola | Co to jest | U nas |
|---|---|---|
| **Host** | aplikacja AI, która zarządza klientami | notebook, Databricks App |
| **Klient** | jedno połączenie z jednym serwerem | `DatabricksMCPClient` |
| **Serwer** | wystawia narzędzia (tools), zasoby (resources) i prompty | zarządzane serwery Databricks |

**Zarządzane serwery MCP** nie wymagają żadnego kodu po stronie serwera. Uprawnienia Unity Catalog nadal obowiązują: klient widzi tylko to, do czego ma `EXECUTE` albo `SELECT`.

| Serwer | Adres | Narzędzia |
|---|---|---|
| Funkcje UC | `/api/2.0/mcp/functions/{catalog}/{schema}` | każda funkcja w schemacie, opis = `COMMENT` |
| AI Search | `/api/2.0/mcp/vector-search/{catalog}/{schema}` | indeksy w schemacie (ścieżka API zachowała starą nazwę) |
| Genie Agent | `/api/2.0/mcp/genie/{space_id}` | pytanie w języku naturalnym → SQL → wynik |
| Zewnętrzne | `/api/2.0/mcp/external/{connection}` | SaaS przez połączenie UC, np. GitHub albo Slack |

**Lab:** pobierz listę narzędzi z serwera funkcji i wywołaj `get_customer_profile` dla klienta X. Na liście zobaczysz też **funkcję z capstone** (`capstone_…`). Zbudowałeś ją w capstone, a dowolny agent zgodny z MCP może jej już użyć bez żadnej integracji. Opis narzędzia to dokładnie `COMMENT` z M2, a wynik jest ten sam co w teście payloadem.

In [ ]:
# source: new + WS4[18]
# ZADANIE 14: wywołaj narzędzie przez MCP.
import nest_asyncio
from databricks_mcp import DatabricksMCPClient

nest_asyncio.apply()  # list_tools() i call_tool() uruchamiają asyncio.run, a notebook ma już pętlę zdarzeń

if not TRY_MCP:
    print("TRY_MCP = False: pomijam.")
else:
    try:
        mcp_client = DatabricksMCPClient(server_url=MCP_SERVERS["funkcje UC"], workspace_client=w)
        mcp_tools = mcp_client.list_tools()
        for tool in mcp_tools:
            print(f"🔧 {tool.name}\n   {(tool.description or '')[:140]}")

        profile_tool = next(tool.name for tool in mcp_tools if tool.name.endswith("get_customer_profile"))
        # TODO: wywołaj profile_tool przez mcp_client.call_tool(nazwa, argumenty);
        #       argumenty to słownik z parametrem funkcji z M2: requested_customer_id = VIP_CUSTOMER_ID
        result = ...
        print(f"\n▶ {profile_tool}({VIP_CUSTOMER_ID}):")
        print("".join(getattr(part, "text", "") for part in result.content))
    except Exception as e:
        print(f"Wywołanie MCP nie powiodło się: {type(e).__name__}: {str(e)[:200]}")
        print("Jeśli to błąd uprawnień albo podglądu, obejrzyj demo prowadzącego. Wzorzec jest ten sam co w M2.")

<!-- source: WS4[17] + slide 60 -->
## 2. Demo: agent z narzędziami z trzech serwerów MCP

Ten sam `SYSTEM_PROMPT` i to samo pytanie „oba” z macierzy tras M5. Tym razem agent nie zna żadnej funkcji z nazwy: pyta serwery MCP „jakie masz narzędzia?” i wywołuje je jednym protokołem.

Agent korzysta z `create_agent` z LangChain 1.x. W LangGraph 1.x `create_react_agent` jest przestarzały, a parametr `state_modifier` z WS4 już nie istnieje. Narzędzia MCP są asynchroniczne, więc agenta wywołujemy przez `ainvoke`.

In [ ]:
# source: WS4[19]
from databricks_langchain import ChatDatabricks, DatabricksMCPServer, DatabricksMultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import ToolMessage

QUESTION = f"Pokaż profil klienta {VIP_CUSTOMER_ID} i co o jego segmencie piszą raporty."


async def ask_mcp_agent(question: str) -> dict:
    client = DatabricksMultiServerMCPClient([
        DatabricksMCPServer(name=label.replace(" ", "-").lower(), url=url, workspace_client=w)
        for label, url in MCP_SERVERS.items()
    ])
    tools = await client.get_tools()
    print(f"Narzędzia z {len(MCP_SERVERS)} serwerów MCP: {[tool.name for tool in tools]}")
    agent = create_agent(
        model=ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.1),
        tools=tools,
        system_prompt=SYSTEM_PROMPT,
    )
    return await agent.ainvoke({"messages": [{"role": "user", "content": question}]})


if not TRY_MCP:
    print("TRY_MCP = False: pomijam.")
else:
    try:
        state = run_async(ask_mcp_agent(QUESTION))
        used = [message.name for message in state["messages"] if isinstance(message, ToolMessage)]
        print(f"\nWywołane narzędzia: {used}\n")
        print(state["messages"][-1].content)
    except Exception as e:
        print(f"Agent MCP niedostępny w tym workspace: {type(e).__name__}: {str(e)[:250]}")

<!-- source: slide 61 + slide 62 + slide 63 + WS2[10] + WS2[13] -->
## 3. Ryzyka agentów: czym różnią się od ryzyk czatu

| Ryzyko | Jak wygląda | Dlaczego u agenta jest gorzej |
|---|---|---|
| **Prompt injection, jailbreak** | „To tylko powieść…”, „zignoruj instrukcje i…”, instrukcja ukryta w dokumencie | czat by to opisał, agent może to **wykonać** |
| **Wyciek PII** | narzędzie zwraca `tax_id`, model go powtarza | ochrona musi być w narzędziu i w danych, nie tylko w prompcie |
| **Zmyślone liczby** | narzędzie zawiodło, model „dopowiada” wynik | wygląda jak wynik z danych i nikt tego nie sprawdzi |
| **Pętle i koszt** | agent woła narzędzia w kółko | 1 pytanie to kilka wywołań modelu; 100 użytkowników to rachunek i limity |
| **Narzędzie z prawem zapisu** | agent tworzy, nadpisuje, wysyła | błąd jest nieodwracalny, a nie tylko błędny |
| **Nadmierne uprawnienia** | agent działa z uprawnieniami twórcy | jeden udany prompt daje dostęp do wszystkiego, co widzisz Ty |

### Sześć warstw obrony, od najtańszej do najtwardszej

| Warstwa | Gdzie działa | Co łapie | Dziś |
|---|---|---|---|
| System prompt | w kodzie agenta | jawnie złe intencje, pytania spoza domeny; da się obejść fikcją | M1, M5 |
| Safety filter Databricks | flaga `enable_safety_filter` w wywołaniu | treści niebezpieczne według polityki platformy; na Free zwracała błędy i jest wypierana przez guardrails na endpoincie | M1 (opcja) |
| Własny guard z taksonomią | drugi model przed i po odpowiedzi | Twoje kategorie: S1 przemoc, S2 przestępstwa, **S3 PII**, S4 nieuczciwe praktyki, S5 nienawiść, S6 samookaleczenie | WS2 (archiwum) |
| **Unity Gateway** (dawniej AI Gateway, GA od 08.2026) | na endpoincie i serwerze MCP, poza kodem | guardrails, blokada PII, limity, logi payloadów, fallback modelu; nie da się ich „zapomnieć” w nowym notebooku | kierunek |
| Narzędzia bez PII | w funkcji UC | agent nie ma czego ujawnić | M2 |
| Row filter, column mask | w Unity Catalog | nawet gdy model zawiedzie, PII nie opuści katalogu | M4 |

> Unity Gateway daje dla MCP także **on-behalf-of execution**: agent działa z uprawnieniami osoby, która pyta, a nie ze wspólnego konta serwisowego.

<!-- source: slide 64 -->
### Least privilege: agent to tożsamość, nie funkcja

- **W notebooku** agent działa z **Twoimi** uprawnieniami: widzi wszystko, co widzisz Ty.
- **W Databricks Apps** działa jako **service principal** aplikacji, z osobnymi `GRANT`-ami.
- Minimum dla naszego agenta: `EXECUTE` na trzech funkcjach i `SELECT` na indeksie. **Żadnego `SELECT` na `gold_customer_360`**: agent dostaje funkcję, a nie tabelę.
- Dwie osobne decyzje: **kto może wywołać agenta** (użytkownicy aplikacji) i **co agent może zrobić** (lista narzędzi i ich uprawnienia).
- Narzędzie z prawem zapisu to osobna zgoda, osobny zakres i osobny log. MCP: `EXECUTE` na konkretnym serwerze, nie na wszystkim.

**Test przed PoC:** wyłącz swoje konto z równania. Czy agent nadal ma dostęp do wszystkiego, czego potrzebuje, i do niczego więcej?

<!-- source: slide 65 + K:Warsztaty_Krzysztof/sprawozdanie_zbiorcze_v3.pdf -->
## 4. Ograniczenia Free Edition i małych środowisk

| Działa | Działa z limitami | Tylko na płatnym workspace |
|---|---|---|
| AI Playground z narzędziami | Serverless: limity czasu i mocy, wyłączenie do końca dnia po przekroczeniu kwoty | Knowledge Assistant |
| funkcje Unity Catalog, row filter, column mask | AI Search: 1 endpoint, 1 jednostka | Unity Gateway na własnym endpoincie |
| Genie Agent | Foundation Model API: limity wywołań na minutę, kolejka przy równoległych wywołaniach | Model Serving własnego agenta |
| agent w notebooku, MLflow Tracing | Genie API: ok. 5 pytań na minutę | Lakehouse Monitoring |
| `ResponsesAgent` lokalnie | Databricks Apps: do 3 aplikacji | secret scope i tokeny serwisowe |
| | zarządzane serwery MCP: Public Preview | reranker AI Search (zablokowany konfiguracją workspace) |
| | | tabele inference i trace'y w Unity Catalog: wymagają katalogu z external storage, a `workspace.default` to default storage |
| | | tworzenie własnego endpointu Model Serving (provisioning kończył się `Failed`) |

Ostatnie trzy wiersze pochodzą z testów na Free Edition opisanych w `Warsztaty_Krzysztof/sprawozdanie_zbiorcze_v3.pdf` (lipiec 2026). Pełna tabela jest w `workshop/docs/cheat_sheet_free_vs_premium.md`. Agent w notebooku i Playground pokazują ok. 90% wzorca, a serving, Gateway i Apps warto zobaczyć na jednym płatnym workspace, zanim zbudujesz własny.

<!-- source: slide 66 + slide 67 + slide 68 + WS2[47] -->
### Cykl życia agenta i co dodać przed PoC oraz produkcją

**Cztery etapy, nie po kolei:** wdrożenie (Databricks Apps + Asset Bundles), obserwowalność (MLflow Tracing, dziś w notebooku), ewaluacja (scorery na trace'ach), monitoring (ocena próbki ruchu na żywo). Tracing włączasz przed wdrożeniem, a scorery podpinasz do działającego agenta.

| Krok | Co | Przed PoC | Przed produkcją |
|---|---|---|---|
| 1. Ewaluacja | zestaw pytań z oczekiwanymi trasami i odpowiedziami; scorery: bezpieczeństwo, PII, poprawność; `mlflow.genai.evaluate` | ✅ | ✅ |
| 2. Monitoring | trace'y i logi payloadów w tabelach; ocena próbki (np. 30%) na żywo; alert na odmowy i toksyczność | ✅ | ✅ |
| 3. Unity Gateway | guardrails, blokada PII, limity i fallback modelu na endpoincie | | ✅ |
| 4. Wdrożenie | Databricks App z service principal; wersje agenta w UC z aliasem `@champion` | | ✅ |
| 5. CI/CD | ewaluacja jako bramka przed wdrożeniem: metryki spadły, deploy zablokowany | | ✅ |

**Pętla zwrotna:** pytania, które w produkcji wypadają źle, trafiają do zestawu offline. Sześć tras z M5 to pierwsze wiersze tego zestawu.

```python
# Ten sam scorer offline i online (wzorzec z WS2 Cz. 3–4)
from mlflow.genai.scorers import Safety, ScorerSamplingConfig
Safety().register(name="retail_safety").start(sampling_config=ScorerSamplingConfig(sample_rate=0.3))
```

<!-- source: slide 69 + slide 70 + K:Warsztaty_Krzysztof/KONTEKST_KONTYNUACJI_PROJEKTU.md -->
## Co potrafisz po dzisiejszym dniu

- [ ] Wyjaśnić różnicę między chatbotem, RAG i agentem, i powiedzieć, kiedy agent **nie** jest potrzebny (M1).
- [ ] Zbudować prosty prototyp agenta na Databricks: w Playground bez kodu i w notebooku z kodem (M1, M5).
- [ ] Użyć RAG jako jednego z narzędzi agenta, obok narzędzia do danych tabelarycznych (M3, M5).
- [ ] Dodać narzędzie do tabeli jako funkcję Unity Catalog, bez PII w wyniku, i przetestować je bez modelu (M2).
- [ ] Przetestować, kiedy agent użyje RAG, funkcji albo fallbacku, i poprawić złą trasę przez opis narzędzia (M5).
- [ ] Wskazać podstawowe ryzyka: bezpieczeństwo (injection, PII, zapis), koszty (wywołania na pytanie) i governance (least privilege) (M4, M6).

**Materiały:** całe repozytorium warsztatu (`workshop/`): notebooki `demo/` i `labs/`, cheat sheet Free vs Premium.

**Dalsza droga (archiwa w repozytorium):**

| Temat | Gdzie | Uwaga |
|---|---|---|
| guardrails z taksonomią, ewaluacja, monitoring | `Warsztaty_Mariusz/` WS2 | ta sama fabuła TechRetail |
| rejestracja agenta, aplikacja | `Warsztaty_Mariusz/` WS4 | wdrożenie przez Model Serving to dziś legacy |
| RAG od zera na innym korpusie, Knowledge Assistant z Examples/Guidelines | `Warsztaty_Krzysztof/rag_agent` 01–05 | angielski korpus robotyki; zweryfikowane na Free 07.2026 |
| agent na funkcjach UC, tracing, tagi trace'ów | `Warsztaty_Krzysztof/single_agent_app` 06–09 | dane Airbnb; zweryfikowane na Free 07.2026 |
| benchmark modeli (ROUGE), LLM-as-a-judge, Llama Guard | `Warsztaty_Krzysztof/genai_eval_and_monitor` 11–14 | klasyczne `mlflow.evaluate`; Llama Guard wymaga endpointu spoza Free |
| batch inference, monitoring odpowiedzi, wariant offline | `Warsztaty_Krzysztof/genai_deploy_and_monitor(_v2)` 15–17 | wariant `_v2` działa bez endpointów |

**Sprzątanie na Free Edition:** endpoint AI Search zużywa kwotę nawet bez zapytań. Po warsztacie usuń go w **Compute → AI Search** albo zostaw, jeśli wracasz jutro.

<!-- source: new + slide 64 + slide 67 -->
## Karta wzorca: od prototypu do PoC na Twoich danych

1. **Narzędzia przez MCP:** funkcje ze schematu są od razu dostępne dla każdego zgodnego agenta (uprawnienia UC obowiązują).
2. **Tożsamość agenta:** service principal z `EXECUTE` na funkcjach i `SELECT` na indeksie, bez dostępu do tabel.
3. **Przed PoC:** zestaw pytań z oczekiwanymi trasami + scorery (bezpieczeństwo, dane wrażliwe, poprawność) + trace'y.
4. **Przed produkcją:** guardrails na endpoincie (Unity Gateway), monitoring, Databricks App, ewaluacja jako bramka CI/CD.

**Canvas agenta** (`workshop/transfer/canvas_agenta.md`): czego brakuje do PoC na Twoich danych: dane, uprawnienia, zestaw testowy, właściciel biznesowy.

<!-- source: new -->
## Podsumowanie

- **MCP** standaryzuje dostęp do narzędzi. Zarządzane serwery Databricks wystawiają funkcje UC, indeksy AI Search i Genie Agenty bez kodu, a uprawnienia Unity Catalog obowiązują dalej.
- Agent może **wykonać**, a nie tylko opisać, więc ryzyka rosną: injection, PII, zmyślone liczby, koszt, zapis, uprawnienia.
- Obrona w głąb: prompt, narzędzie bez PII, maska w katalogu, a przed produkcją Unity Gateway, ewaluacja jako bramka i monitoring.
- Agent to **tożsamość**: minimum uprawnień, osobny service principal, `EXECUTE` zamiast `SELECT` na tabeli.

Dziękujemy! Pytania?